# Exercice 1 : Mise en place de Kafka et dʼun producteur simple
afka"

1.1 Créez un topic Kafka nommé "weather_stream"

In [1]:
from kafka.admin import KafkaAdminClient, NewTopic

admin = KafkaAdminClient(bootstrap_servers="kafka:9092")

topic = NewTopic(name="weather_stream", num_partitions=1, replication_factor=1)
admin.create_topics([topic])

print(admin.list_topics())


['weather_stream']


1.2 Envoyez un message statique, par exemple :
{"msg": "Hello Kafka"}

In [3]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

message = {"msg": "Hello Kafka"}
resultat = producer.send("weather_stream", message).get()
producer.flush()

print("Message envoyé :", message)
print("Topic :", resultat.topic, "| partition :", resultat.partition, "| offset :", resultat.offset)


Message envoyé : {'msg': 'Hello Kafka'}
Topic : weather_stream | partition : 0 | offset : 0


# Exercice 2 : Écriture dʼun consommateur Kafka

## 2.1  et 2.2 Créez un script Python consommateur qui lit les messages depuis un topic Kafka défini en
variable au début du scrip et affciher en tels rééelt.

In [9]:
from kafka import KafkaConsumer
import json

TOPIC = "weather_stream"

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    value_deserializer=lambda v: v.decode("utf-8"),
    consumer_timeout_ms=60000
)

print("En écoute sur", TOPIC, "...")
for message in consumer:
    texte = message.value
    try:
        donnees = json.loads(texte)
        print("offset", message.offset, ":", donnees)
    except json.JSONDecodeError:
        print("offset", message.offset, ": message ignoré (pas du JSON) ->", repr(texte))

print("Plus de message depuis 60 secondes, fin de l'écoute.")
consumer.close()


En écoute sur weather_stream ...
offset 0 : {'msg': 'Hello Kafka'}
offset 1 : message ignoré (pas du JSON) -> ''
offset 2 : {'msg': 'test temps reel'}
offset 3 : message ignoré (pas du JSON) -> ' {\'msg\': \'enmsg ilham"}'
offset 4 : message ignoré (pas du JSON) -> '{\'msg\': "message ilham cette fois comme il faut"}'
offset 5 : {'msg': 'je vais y arriver'}
offset 6 : {'msg': 'yes'}
Plus de message depuis 60 secondes, fin de l'écoute.
